<a href="https://colab.research.google.com/github/ingbabic/python/blob/main/Celka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("expression_data.csv")
df.head()

,Gene_Symbol,WT_Rep1,WT_Rep2,WT_Rep3,MUT_Rep1,MUT_Rep2,MUT_Rep3
0,TP53,2926,2396,2284,561,731,679
1,BRCA1,2003,2190,2473,2000,2170,2615
2,EGFR,2235,2079,2619,2518,1971,2211
3,MYC,1405,1110,994,249,338,309
4,PTEN,230,269,215,231,256,237


In [2]:
print("Rows and columns:", df.shape)
print("Column names:", df.columns.tolist())

Rows and columns: (1000, 7)
Column names: ['Gene_Symbol', 'WT_Rep1', 'WT_Rep2', 'WT_Rep3', 'MUT_Rep1', 'MUT_Rep2', 'MUT_Rep3']


In [3]:
wt_columns = ["WT_Rep1", "WT_Rep2", "WT_Rep3"]
mut_columns = ["MUT_Rep1", "MUT_Rep2", "MUT_Rep3"]

df["Mean_WT"] = df[wt_columns].mean(axis=1)
df["Mean_MUT"] = df[mut_columns].mean(axis=1)

df[["Gene_Symbol", "Mean_WT", "Mean_MUT"]].head()

,Gene_Symbol,Mean_WT,Mean_MUT
0,TP53,2535.333333,657.000000
1,BRCA1,2222.000000,2261.666667
2,EGFR,2311.000000,2233.333333
3,MYC,1169.666667,298.666667
4,PTEN,238.000000,241.333333


In [4]:
import numpy as np

df["Log2FC"] = np.log2(df["Mean_MUT"] / df["Mean_WT"])

df[["Gene_Symbol", "Mean_WT", "Mean_MUT", "Log2FC"]].head()

,Gene_Symbol,Mean_WT,Mean_MUT,Log2FC
0,TP53,2535.333333,657.000000,-1.948210
1,BRCA1,2222.000000,2261.666667,0.025527
2,EGFR,2311.000000,2233.333333,-0.049319
3,MYC,1169.666667,298.666667,-1.969489
4,PTEN,238.000000,241.333333,0.020066


In [5]:
upregulated = df[df["Log2FC"] > 1.5]
downregulated = df[df["Log2FC"] < -1.5]

print("Upregulated genes:", len(upregulated))
print("Downregulated genes:", len(downregulated))

Upregulated genes: 105
Downregulated genes: 105


In [6]:
all_replicate_columns = wt_columns + mut_columns

df["Average_Expression"] = df[all_replicate_columns].mean(axis=1)

df["Regulation"] = np.select(
    [df["Log2FC"] > 1.5, df["Log2FC"] < -1.5],
    ["Upregulated", "Downregulated"],
    default="Unchanged"
)

df[["Gene_Symbol", "Average_Expression", "Log2FC", "Regulation"]].head()

,Gene_Symbol,Average_Expression,Log2FC,Regulation
0,TP53,1596.166667,-1.948210,Downregulated
1,BRCA1,2241.833333,0.025527,Unchanged
2,EGFR,2272.166667,-0.049319,Unchanged
3,MYC,734.166667,-1.969489,Downregulated
4,PTEN,239.666667,0.020066,Unchanged


In [7]:
import plotly.express as px

fig = px.scatter(
    df,
    x="Average_Expression",
    y="Log2FC",
    color="Regulation",
    hover_name="Gene_Symbol",
    log_x=True,
    color_discrete_map={
        "Upregulated": "red",
        "Downregulated": "blue",
        "Unchanged": "gray"
    },
    title="Gene Expression: Mutant vs Wild-Type"
)

fig.update_layout(
    xaxis_title="Average expression across all samples (log scale)",
    yaxis_title="Log2 fold change (MUT / WT)"
)

fig.show()